# RLHF from Scratch: Base Model → SFT → Reward Model → PPO

This notebook walks through the full InstructGPT-style pipeline on GPT-2 (124M), small enough to run on a MacBook:

| Stage | What happens | Output |
|---|---|---|
| **0. Base model** | Pretrained next-token predictor | Rambles, doesn't follow instructions |
| **1. SFT** (supervised fine-tuning) | Train on human-written (instruction, response) pairs from Dolly | A model that answers in the right *shape* |
| **2. Reward model** | Train on human *preferences* ("A is better than B") with a pairwise Bradley-Terry loss | A function `r(prompt, response) → scalar` |
| **3. RL (PPO)** | Optimise the SFT model to maximise `r`, with a KL penalty to a frozen SFT copy | A model humans (as modelled by `r`) prefer |

The reward model and PPO loop are written in plain PyTorch, with no RL library, so every step is visible.

In [ ]:
import itertools
import math
import os
import random

# Fall back to CPU for any op the Apple GPU (MPS) backend doesn't implement yet.
os.environ["PYTORCH_ENABLE_MPS_FALLBACK"] = "1"
# TensorFlow may be installed in the venv; stop transformers from importing it
os.environ["USE_TF"] = "0"
os.environ["USE_TORCH"] = "1"

import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from datasets import load_dataset
from transformers import (
    AutoModel,
    AutoModelForCausalLM,
    AutoTokenizer,
    Trainer,
    TrainingArguments,
)

In [ ]:
BASE_MODEL = "gpt2"
SFT_DIR = "sft-gpt2-dolly"   # where the SFT model is saved; re-running skips SFT if it exists
SEED = 42

if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"


def set_seed(seed):
    random.seed(seed)
    torch.manual_seed(seed)


set_seed(SEED)
print("device:", device)

### Tokenizers and the prompt template

GPT-2 has no padding token, so we reuse EOS (`<|endoftext|>`). We need two padding directions:

- **Right padding** for training and scoring: real tokens first, padding after.
- **Left padding** for batched generation: every prompt must *end* at the same column so new tokens are appended right after it.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)                           # right padding
gen_tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, padding_side="left")  # left padding
tokenizer.pad_token = tokenizer.eos_token
gen_tokenizer.pad_token = gen_tokenizer.eos_token
EOS = tokenizer.eos_token_id


def build_prompt(instruction, context=""):
    prompt = (
        "Below is an instruction that describes a task. "
        "Write a response that appropriately completes the request.\n\n"
        f"### Instruction:\n{instruction.strip()}\n\n"
    )
    if context and context.strip():
        prompt += f"### Context:\n{context.strip()}\n\n"
    prompt += "### Response:\n"
    return prompt


@torch.no_grad()
def generate(model, instruction, context="", max_new_tokens=100):
    # Greedy decoding: deterministic, so before/after comparisons are fair.
    inputs = tokenizer(build_prompt(instruction, context), return_tensors="pt").to(device)
    model.eval()
    out = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        repetition_penalty=1.1,
        pad_token_id=EOS,
        eos_token_id=EOS,
    )
    return tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()


@torch.no_grad()
def sample(model, instructions, max_new_tokens=64, **gen_kwargs):
    # Batched sampling (left-padded). Returns one decoded response per instruction.
    enc = gen_tokenizer([build_prompt(i) for i in instructions], return_tensors="pt", padding=True).to(device)
    model.eval()
    out = model.generate(
        **enc, do_sample=True, max_new_tokens=max_new_tokens, pad_token_id=EOS, eos_token_id=EOS, **gen_kwargs
    )
    return [tokenizer.decode(o[enc["input_ids"].shape[1]:], skip_special_tokens=True).strip() for o in out]

In [ ]:
test_prompts = [
    "What is the capital of France?",
    "Give me three tips for staying healthy.",
    "How do I make a cup of tea?",
    "Why is the sky blue?",
    "Write a polite email declining a meeting invitation.",
]
results = {p: {} for p in test_prompts}   # results[prompt][stage] = response

## Stage 0: The base model

GPT-2 was only trained to continue web text. Given an instruction, it continues the *document* rather than answering it.

In [ ]:
base_model = AutoModelForCausalLM.from_pretrained(BASE_MODEL).to(device)
print(f"parameters: {base_model.num_parameters() / 1e6:.0f}M")

for p in test_prompts:
    results[p]["base"] = generate(base_model, p)
    print("=" * 80, "\nINSTRUCTION:", p, "\n" + "-" * 80)
    print(results[p]["base"])

del base_model

## Stage 1: Supervised fine-tuning (SFT) on Dolly

This is the same recipe as `../instruction-fine-tuning`: train on `prompt + response + <eos>`, with the loss only on the response tokens (prompt labels are `-100`). SFT teaches the model the *format* of a helpful answer and when to stop.

The SFT model is the starting point for everything that follows: it is the policy we'll optimise, the initialisation of the reward model, and the frozen reference for the KL penalty.

In [ ]:
SFT_MAX_LEN = 512
SFT_NUM_TRAIN = 5000   # None for all ~15k


def tokenize_sft(example):
    prompt_ids = tokenizer(build_prompt(example["instruction"], example["context"]))["input_ids"]
    response_ids = tokenizer(example["response"].strip())["input_ids"] + [EOS]
    input_ids = (prompt_ids + response_ids)[:SFT_MAX_LEN]
    labels = ([-100] * len(prompt_ids) + response_ids)[:SFT_MAX_LEN]
    return {"input_ids": input_ids, "attention_mask": [1] * len(input_ids), "labels": labels}


def collate_sft(batch):
    max_len = max(len(ex["input_ids"]) for ex in batch)
    pad = lambda seq, value: seq + [value] * (max_len - len(seq))
    return {
        "input_ids": torch.tensor([pad(ex["input_ids"], EOS) for ex in batch]),
        "attention_mask": torch.tensor([pad(ex["attention_mask"], 0) for ex in batch]),
        "labels": torch.tensor([pad(ex["labels"], -100) for ex in batch]),
    }


if os.path.isdir(SFT_DIR):
    print(f"Found {SFT_DIR}/, skipping SFT training")
else:
    dolly = load_dataset("databricks/databricks-dolly-15k", split="train").shuffle(seed=SEED)
    if SFT_NUM_TRAIN is not None:
        dolly = dolly.select(range(SFT_NUM_TRAIN))
    sft_ds = dolly.map(tokenize_sft, remove_columns=dolly.column_names)
    sft_ds = sft_ds.filter(lambda ex: any(l != -100 for l in ex["labels"]))

    sft_trainer = Trainer(
        model=AutoModelForCausalLM.from_pretrained(BASE_MODEL).to(device),
        args=TrainingArguments(
            output_dir="sft-checkpoints",
            num_train_epochs=2,
            per_device_train_batch_size=8,
            gradient_accumulation_steps=2,
            learning_rate=5e-5,
            lr_scheduler_type="cosine",
            warmup_ratio=0.03,
            weight_decay=0.01,
            logging_steps=50,
            save_strategy="no",
            report_to="none",
            group_by_length=True,
            dataloader_pin_memory=False,   # pinned memory is CUDA-only
            seed=SEED,
        ),
        train_dataset=sft_ds,
        data_collator=collate_sft,
    )
    sft_trainer.train()
    sft_trainer.save_model(SFT_DIR)
    tokenizer.save_pretrained(SFT_DIR)
    del sft_trainer

In [ ]:
sft_model = AutoModelForCausalLM.from_pretrained(SFT_DIR).to(device)

for p in test_prompts:
    results[p]["sft"] = generate(sft_model, p)
    print("=" * 80, "\nINSTRUCTION:", p)
    print("-" * 37, "BASE", "-" * 37)
    print(results[p]["base"])
    print("-" * 37, "SFT", "-" * 38)
    print(results[p]["sft"])

## Stage 2: Reward model

SFT can only imitate the demonstrations it was given. But for most prompts it's much easier for a person to say *which of two answers is better* than to write the perfect answer. The reward model turns those comparisons into a score.

### 2a. Sample several responses per prompt from the SFT model

In InstructGPT, labellers were shown K responses (4 to 9) from the SFT model for each prompt and ranked them. Here are K = 4 samples per prompt. Notice they vary in quality: that variation is what the labels capture.

In [ ]:
K = 4
label_prompts = [
    "What are some good ways to save money?",
    "Explain photosynthesis in simple terms.",
    "What should I pack for a weekend camping trip?",
    "Why do cats purr?",
]

set_seed(SEED)
sft_samples = {}
for p in label_prompts:
    sft_samples[p] = sample(sft_model, [p] * K, temperature=0.9, top_p=0.95)
    print("=" * 80, "\nPROMPT:", p)
    for i, r in enumerate(sft_samples[p]):
        print(f"\n[{chr(65 + i)}] {r}")

### 2b. Human preference labels

You can be the labeller. Set `LABEL_MYSELF = True` and, for each prompt, you'll be shown pairs of the samples above and asked which is better.

A few dozen labels are nowhere near enough to train a reward model, so we'll also use **Anthropic's HH-RLHF** dataset: about 160k comparisons where crowdworkers chatted with a model, were shown two responses, and picked the more helpful one. Your own labels (if any) are added to the training set.

One caveat: HH-RLHF responses came from Anthropic's models, not from *our* SFT model. In a real pipeline the comparisons are on the policy's own samples, so the reward model is accurate exactly where the policy will be generating. Here there's a distribution mismatch, which is one more reason to keep the policy close to SFT (see the KL penalty below).

In [ ]:
LABEL_MYSELF = False   # set True to label pairs interactively

my_pairs = []
if LABEL_MYSELF:
    stop = False
    for p, responses in sft_samples.items():
        for i, j in itertools.combinations(range(K), 2):
            print("=" * 80, "\nPROMPT:", p)
            print(f"\n[A] {responses[i]}\n\n[B] {responses[j]}\n")
            choice = input("Which is better? a / b / s(kip) / q(uit): ").strip().lower()
            if choice == "q":
                stop = True
                break
            if choice in ("a", "b"):
                chosen, rejected = (responses[i], responses[j]) if choice == "a" else (responses[j], responses[i])
                my_pairs.append({"prompt": p, "chosen": chosen, "rejected": rejected})
        if stop:
            break

print(f"{len(my_pairs)} pairs labelled by you")

### 2c. Load HH-RLHF preference pairs

Each row has a `chosen` and a `rejected` conversation in the form `\n\nHuman: ...\n\nAssistant: ...`. We keep single-turn conversations from the *helpful* subset and convert them to our instruction template, giving `(prompt, chosen, rejected)` triples.

In [ ]:
RM_MAX_LEN = 384


def parse_single_turn(text):
    if text.count("\n\nHuman:") != 1 or text.count("\n\nAssistant:") != 1:
        return None
    human, assistant = text.split("\n\nAssistant:")
    return human.replace("\n\nHuman:", "").strip(), assistant.strip()


def to_pairs(split):
    pairs = []
    for ex in split:
        c, r = parse_single_turn(ex["chosen"]), parse_single_turn(ex["rejected"])
        if c is None or r is None or c[0] != r[0]:
            continue
        prompt, chosen, rejected = c[0], c[1], r[1]
        if chosen and rejected and chosen != rejected:
            pairs.append({"prompt": prompt, "chosen": chosen, "rejected": rejected})
    return pairs


hh = load_dataset("Anthropic/hh-rlhf", data_dir="helpful-base")
hh_train, hh_test = to_pairs(hh["train"]), to_pairs(hh["test"])
print(f"single-turn pairs: {len(hh_train)} train / {len(hh_test)} test")
print(hh_train[0])

In [ ]:
def encode(prompt, response):
    # prompt + response + <eos>. The reward is read off the final (EOS) position.
    ids = tokenizer(build_prompt(prompt))["input_ids"] + tokenizer(response)["input_ids"] + [EOS]
    return ids[-RM_MAX_LEN:]   # if too long, drop from the left so the EOS survives


def fits(pair):
    n_prompt = len(tokenizer(build_prompt(pair["prompt"]))["input_ids"])
    n_resp = max(len(tokenizer(pair["chosen"])["input_ids"]), len(tokenizer(pair["rejected"])["input_ids"]))
    return n_prompt + n_resp + 1 <= RM_MAX_LEN


RM_NUM_TRAIN = 10000
RM_NUM_EVAL = 500

random.Random(SEED).shuffle(hh_train)
rm_train_pairs = [p for p in hh_train if fits(p)][:RM_NUM_TRAIN] + my_pairs
rm_eval_pairs = [p for p in hh_test if fits(p)][:RM_NUM_EVAL]
print(f"{len(rm_train_pairs)} train pairs / {len(rm_eval_pairs)} eval pairs")

### 2d. The reward model: SFT backbone + scalar head

We start from the SFT model's transformer (it already understands the prompt format and what answers look like), drop the language-model head, and add a linear layer that maps the final hidden state to a single number.

**Bradley-Terry loss.** Model the probability that a human prefers response $y_w$ over $y_l$ as

$$P(y_w \succ y_l \mid x) = \sigma\big(r(x, y_w) - r(x, y_l)\big)$$

and maximise its log-likelihood over the labelled pairs:

$$\mathcal{L} = -\log \sigma\big(r(x, y_w) - r(x, y_l)\big)$$

Only the *difference* in rewards matters, so the absolute scale is arbitrary (we normalise it before PPO).

In [ ]:
class RewardModel(nn.Module):
    def __init__(self, path):
        super().__init__()
        self.backbone = AutoModel.from_pretrained(path)   # GPT2Model: transformer without the LM head
        self.head = nn.Linear(self.backbone.config.n_embd, 1)

    def forward(self, input_ids, attention_mask):
        hidden = self.backbone(input_ids=input_ids, attention_mask=attention_mask).last_hidden_state
        last = attention_mask.sum(dim=1) - 1   # index of the final real token (right padding)
        return self.head(hidden[torch.arange(len(last), device=last.device), last]).squeeze(-1)


def pad_right(seqs):
    max_len = max(len(s) for s in seqs)
    ids = torch.tensor([s + [EOS] * (max_len - len(s)) for s in seqs])
    mask = torch.tensor([[1] * len(s) + [0] * (max_len - len(s)) for s in seqs])
    return ids.to(device), mask.to(device)


def pair_rewards(model, pairs):
    # One forward pass over [chosen_1..chosen_B, rejected_1..rejected_B]
    seqs = [encode(p["prompt"], p["chosen"]) for p in pairs] + [encode(p["prompt"], p["rejected"]) for p in pairs]
    rewards = model(*pad_right(seqs))
    return rewards[: len(pairs)], rewards[len(pairs):]


@torch.no_grad()
def evaluate_rm(model, pairs, batch_size=16):
    model.eval()
    correct, margins = 0, []
    for i in range(0, len(pairs), batch_size):
        r_c, r_r = pair_rewards(model, pairs[i : i + batch_size])
        correct += (r_c > r_r).sum().item()
        margins += (r_c - r_r).tolist()
    return correct / len(pairs), sum(margins) / len(margins)


reward_model = RewardModel(SFT_DIR).to(device)
acc, margin = evaluate_rm(reward_model, rm_eval_pairs)
print(f"before training: accuracy {acc:.1%}, mean margin {margin:+.3f}   (random head, so ~50%)")

In [ ]:
RM_BATCH = 8
RM_LR = 1e-5
RM_EPOCHS = 1   # InstructGPT trained 1 epoch; the reward model overfits quickly

optimizer = torch.optim.AdamW(reward_model.parameters(), lr=RM_LR)
rm_log = []
step = 0
for epoch in range(RM_EPOCHS):
    random.Random(SEED + epoch).shuffle(rm_train_pairs)
    for i in range(0, len(rm_train_pairs), RM_BATCH):
        reward_model.train()
        r_c, r_r = pair_rewards(reward_model, rm_train_pairs[i : i + RM_BATCH])
        loss = -F.logsigmoid(r_c - r_r).mean()   # Bradley-Terry

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(reward_model.parameters(), 1.0)
        optimizer.step()

        rm_log.append({"step": step, "loss": loss.item(), "acc": (r_c > r_r).float().mean().item()})
        if step % 100 == 0:
            recent = rm_log[-100:]
            print(f"step {step:5d}  loss {sum(l['loss'] for l in recent) / len(recent):.3f}  "
                  f"train acc {sum(l['acc'] for l in recent) / len(recent):.1%}")
        step += 1

acc, margin = evaluate_rm(reward_model, rm_eval_pairs)
print(f"\nafter training: eval accuracy {acc:.1%}, mean margin {margin:+.3f}")

Expect eval accuracy somewhere in the 60s. That sounds low, but preference data is noisy: human labellers only agree with *each other* about 65 to 75% of the time on data like this, and GPT-2 small is a weak judge. A reward model doesn't need to be perfect to be a useful training signal, but its errors are exactly what PPO will try to exploit.

In [ ]:
# Plot a smoothed training curve. Colours: categorical slot 1 (blue), 2px line, recessive grid.
def smooth(xs, window=50):
    return [sum(xs[max(0, i - window + 1) : i + 1]) / len(xs[max(0, i - window + 1) : i + 1]) for i in range(len(xs))]


fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 3.5))
steps = [l["step"] for l in rm_log]
ax1.plot(steps, smooth([l["loss"] for l in rm_log]), color="#2a78d6", linewidth=2)
ax1.set_title("Reward model loss (Bradley-Terry)")
ax2.plot(steps, smooth([l["acc"] for l in rm_log]), color="#2a78d6", linewidth=2)
ax2.axhline(0.5, color="#52514e", linewidth=1, linestyle="--")
ax2.annotate("chance", xy=(steps[-1], 0.5), xytext=(0, 4), textcoords="offset points", ha="right", color="#52514e")
ax2.set_title("Reward model train accuracy")
for ax in (ax1, ax2):
    ax.set_xlabel("step")
    ax.grid(alpha=0.25)
    ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.show()

### 2e. The reward model scores any (prompt, response) pair

Now we have a function. Let's rank the SFT samples from step 2a, and try a few hand-written responses to see what it rewards.

In [ ]:
reward_model.eval()
for p in reward_model.parameters():
    p.requires_grad_(False)   # frozen from here on


@torch.no_grad()
def rm_score(prompts, responses, batch_size=16):
    scores = []
    for i in range(0, len(prompts), batch_size):
        seqs = [encode(p, r) for p, r in zip(prompts[i : i + batch_size], responses[i : i + batch_size])]
        scores.append(reward_model(*pad_right(seqs)))
    return torch.cat(scores)


for p, responses in sft_samples.items():
    scores = rm_score([p] * len(responses), responses)
    print("=" * 80, "\nPROMPT:", p)
    for s, r in sorted(zip(scores.tolist(), responses), reverse=True):
        print(f"  {s:+.2f}  {r[:110]!r}")

probe = "Why is the sky blue?"
probe_responses = [
    "The sky is blue because air molecules scatter short (blue) wavelengths of sunlight much more than long (red) ones.",
    "I don't know.",
    "Blue blue blue blue blue blue blue blue blue.",
    "The sky is green.",
]
print("=" * 80, "\nPROBE:", probe)
for s, r in zip(rm_score([probe] * len(probe_responses), probe_responses).tolist(), probe_responses):
    print(f"  {s:+.2f}  {r!r}")

## Stage 3: RL optimisation with PPO

Now treat the SFT model as a **policy** $\pi_\theta$: the *state* is the prompt plus the tokens generated so far, an *action* is the next token, and an *episode* is one full response. At the end of the episode the reward model hands out a score.

### The objective, and why the KL penalty matters

We maximise

$$\mathbb{E}_{x,\, y \sim \pi_\theta}\Big[\, r_\phi(x, y) \;-\; \beta \,\mathrm{KL}\big(\pi_\theta(\cdot \mid x)\ \|\ \pi_{\text{SFT}}(\cdot \mid x)\big) \Big]$$

where $\pi_{\text{SFT}}$ is a **frozen copy** of the SFT model.

The reward model is only a rough approximation of human judgement, trained on a limited slice of text. Push hard enough on it and the policy finds inputs where it's *wrong*: odd phrasings, repeated tokens, or lengths that happen to score high. That's **reward hacking**: reward goes up while real quality goes down. The KL term charges the policy for every bit it drifts from the SFT distribution, so it can only change its behaviour where the reward gain is worth it. It also keeps the policy in the region where the reward model was trained and is actually accurate.

In practice the KL is applied **per token**. For each generated token $t$:

$$r_t = -\beta \big(\log \pi_\theta(y_t \mid \cdot) - \log \pi_{\text{SFT}}(y_t \mid \cdot)\big) \;+\; \mathbb{1}[t = \text{last}] \cdot r_\phi(x, y)$$

### PPO in four steps (repeated every iteration)

1. **Rollout**: sample responses from the current policy for a batch of prompts. Record each token's log-prob under the policy and under the frozen reference, and the value head's estimate.
2. **Reward**: score each full response with the reward model; build the per-token rewards above.
3. **Advantage**: estimate how much better each token was than expected, using the value head and GAE (Generalised Advantage Estimation).
4. **Update**: a few epochs of gradient steps on the clipped PPO objective. Clipping the probability ratio to $[1-\epsilon, 1+\epsilon]$ stops any single update from moving the policy too far.

In [ ]:
# Prompts for RL: single-turn questions from the HH-RLHF test split (not used to train the reward model).
PROMPT_MAX_TOKENS = 96
ppo_prompts = sorted({p["prompt"] for p in hh_test[RM_NUM_EVAL:]})
ppo_prompts = [p for p in ppo_prompts if len(tokenizer(build_prompt(p))["input_ids"]) <= PROMPT_MAX_TOKENS]
print(len(ppo_prompts), "RL prompts, e.g.", random.Random(SEED).sample(ppo_prompts, 3))

### Policy, value head, and the frozen reference

- **Policy**: the SFT model, which we'll update. We add a **value head** (a linear layer on the final hidden state) that predicts the expected future reward from each position. It shares the transformer with the policy, as in most RLHF implementations.
- **Reference**: a second copy of the SFT model with gradients off. It never changes.
- **Reward model**: frozen, from Stage 2.

Dropout is disabled in the policy and reference. With dropout on, the same tokens would get different log-probs on every pass, and the PPO probability ratio would be noise.

In [ ]:
NO_DROPOUT = dict(resid_pdrop=0.0, embd_pdrop=0.0, attn_pdrop=0.0)


class PolicyWithValueHead(nn.Module):
    def __init__(self, path):
        super().__init__()
        self.lm = AutoModelForCausalLM.from_pretrained(path, **NO_DROPOUT)
        self.v_head = nn.Linear(self.lm.config.n_embd, 1)
        nn.init.zeros_(self.v_head.weight)   # start by predicting 0 everywhere
        nn.init.zeros_(self.v_head.bias)


ref_model = AutoModelForCausalLM.from_pretrained(SFT_DIR, **NO_DROPOUT).to(device).eval()
for p in ref_model.parameters():
    p.requires_grad_(False)

del sft_model   # the reference is the same weights

In [ ]:
def response_logprobs(lm, input_ids, attention_mask, prompt_len, v_head=None):
    # Log-prob of each response token (and optionally the value estimate at each step).
    # Position ids come from the attention mask, so left-padded prompts get the same
    # positions they had during generation.
    position_ids = (attention_mask.cumsum(-1) - 1).clamp(min=0)
    out = lm(
        input_ids=input_ids,
        attention_mask=attention_mask,
        position_ids=position_ids,
        output_hidden_states=v_head is not None,
    )
    # logits at position t predict token t+1, so response token i is predicted from position prompt_len-1+i
    logits = out.logits[:, prompt_len - 1 : -1].float()
    logp = torch.log_softmax(logits, dim=-1).gather(-1, input_ids[:, prompt_len:].unsqueeze(-1)).squeeze(-1)
    if v_head is None:
        return logp
    values = v_head(out.hidden_states[-1][:, prompt_len - 1 : -1]).squeeze(-1)
    return logp, values


def masked_mean(x, mask):
    return (x * mask).sum() / mask.sum()


def whiten(x, mask):
    mean = masked_mean(x, mask)
    var = masked_mean((x - mean) ** 2, mask)
    return (x - mean) * torch.rsqrt(var + 1e-8) * mask

### Step 1: Rollout

Sampling uses `temperature=1`, `top_k=0`, `top_p=1`: PPO's maths assumes we sample from the policy's *actual* distribution, so no truncation tricks.

The response mask marks the real tokens: everything up to and including the first EOS. After that, `generate` just pads.

In [ ]:
RESPONSE_LEN = 48


@torch.no_grad()
def rollout(policy, prompts):
    enc = gen_tokenizer([build_prompt(p) for p in prompts], return_tensors="pt", padding=True).to(device)
    prompt_len = enc["input_ids"].shape[1]

    policy.eval()
    seq = policy.lm.generate(
        **enc,
        do_sample=True, temperature=1.0, top_k=0, top_p=1.0,
        max_new_tokens=RESPONSE_LEN, pad_token_id=EOS, eos_token_id=EOS,
    )
    resp = seq[:, prompt_len:]
    is_eos = (resp == EOS).long()
    resp_mask = ((is_eos.cumsum(dim=1) - is_eos) == 0).long()   # 1 up to and including the first EOS
    attn = torch.cat([enc["attention_mask"], resp_mask], dim=1)

    logp, values = response_logprobs(policy.lm, seq, attn, prompt_len, policy.v_head)
    ref_logp = response_logprobs(ref_model, seq, attn, prompt_len)

    responses = [tokenizer.decode(r[m.bool()], skip_special_tokens=True).strip() for r, m in zip(resp, resp_mask)]
    scores = rm_score(prompts, responses)

    return {
        "seq": seq, "attn": attn, "prompt_len": prompt_len, "mask": resp_mask.float(),
        "logp": logp, "values": values, "ref_logp": ref_logp,
        "scores": scores, "prompts": prompts, "responses": responses,
    }

### Reward normalisation

The reward model's scale is arbitrary (only differences were trained). We shift and scale it using the scores of SFT samples, so an average SFT response gets reward ≈ 0 and the KL coefficient $\beta$ has a consistent meaning.

In [ ]:
set_seed(SEED)
calib_prompts = random.sample(ppo_prompts, 128)
calib_responses = []
for i in range(0, len(calib_prompts), 32):
    calib_responses += sample(ref_model, calib_prompts[i : i + 32], max_new_tokens=RESPONSE_LEN, top_k=0, top_p=1.0)
calib_scores = rm_score(calib_prompts, calib_responses)
RM_MEAN, RM_STD = calib_scores.mean().item(), calib_scores.std().item()
print(f"SFT reward: mean {RM_MEAN:+.3f}, std {RM_STD:.3f}")

### Steps 2 and 3: Per-token rewards, then advantages with GAE

The value head predicts $V(s_t)$, the expected future reward from token $t$. The TD error $\delta_t = r_t + \gamma V(s_{t+1}) - V(s_t)$ says whether token $t$ went better or worse than expected. GAE blends TD errors over future steps with decay $\lambda$ to trade off bias and variance:

$$A_t = \sum_{l \ge 0} (\gamma\lambda)^l\, \delta_{t+l}$$

The value target (the "return") is $A_t + V(s_t)$.

In [ ]:
GAMMA = 1.0   # no discounting within a short response
LAM = 0.95


def compute_advantages(r, kl_coef):
    mask = r["mask"]
    kl = (r["logp"] - r["ref_logp"]) * mask   # per-token KL estimate vs the frozen SFT model
    rewards = -kl_coef * kl
    last = mask.sum(dim=1).long() - 1
    rewards[torch.arange(len(last), device=last.device), last] += (r["scores"] - RM_MEAN) / RM_STD   # RM score on the final token

    values = r["values"] * mask
    advantages = torch.zeros_like(rewards)
    last_gae = torch.zeros(len(rewards), device=rewards.device)
    T = rewards.shape[1]
    for t in reversed(range(T)):
        next_value = values[:, t + 1] if t + 1 < T else 0.0
        delta = rewards[:, t] + GAMMA * next_value - values[:, t]
        last_gae = delta + GAMMA * LAM * last_gae
        advantages[:, t] = last_gae
    returns = (advantages + values) * mask
    return whiten(advantages, mask), returns, kl.sum(dim=1)

### Step 4: The PPO update

For each minibatch we recompute log-probs and values under the *current* weights and compare them with the rollout's:

- **Policy loss**: with ratio $\rho_t = \pi_\theta(y_t) / \pi_{\text{old}}(y_t)$, minimise $\max\big(-A_t \rho_t,\ -A_t\, \text{clip}(\rho_t, 1-\epsilon, 1+\epsilon)\big)$. Tokens with positive advantage become more likely and negative ones less likely, but never by more than $\epsilon$ per rollout.
- **Value loss**: regress $V$ towards the returns, also clipped.

In [ ]:
PPO_EPOCHS = 4
MINIBATCH = 8
CLIP_EPS = 0.2
CLIP_VALUE = 0.2
VF_COEF = 0.1


def ppo_update(policy, optimizer, r, advantages, returns):
    policy.train()
    B = r["seq"].shape[0]
    stats = []
    for _ in range(PPO_EPOCHS):
        for idx in torch.randperm(B, device=device).split(MINIBATCH):
            mask = r["mask"][idx]
            logp, values = response_logprobs(policy.lm, r["seq"][idx], r["attn"][idx], r["prompt_len"], policy.v_head)

            ratio = torch.exp((logp - r["logp"][idx]) * mask)   # masked, so padding can't overflow to inf
            adv = advantages[idx]
            pg_loss = masked_mean(torch.max(-adv * ratio, -adv * ratio.clamp(1 - CLIP_EPS, 1 + CLIP_EPS)), mask)

            old_values = r["values"][idx]
            values_clipped = old_values + (values - old_values).clamp(-CLIP_VALUE, CLIP_VALUE)
            ret = returns[idx]
            vf_loss = 0.5 * masked_mean(torch.max((values - ret) ** 2, (values_clipped - ret) ** 2), mask)

            loss = pg_loss + VF_COEF * vf_loss
            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(policy.parameters(), 1.0)
            optimizer.step()
            stats.append({"pg_loss": pg_loss.item(), "vf_loss": vf_loss.item(),
                          "clip_frac": masked_mean(((ratio - 1).abs() > CLIP_EPS).float(), mask).item()})
    return {k: sum(s[k] for s in stats) / len(stats) for k in stats[0]}

In [ ]:
PPO_BATCH = 32
PPO_LR = 1.41e-5


def run_ppo(kl_coef, num_steps, seed=SEED, print_every=10):
    set_seed(seed)
    policy = PolicyWithValueHead(SFT_DIR).to(device)
    optimizer = torch.optim.AdamW(policy.parameters(), lr=PPO_LR)
    history = []
    for step in range(num_steps):
        prompts = random.sample(ppo_prompts, PPO_BATCH)
        r = rollout(policy, prompts)
        advantages, returns, kl = compute_advantages(r, kl_coef)
        stats = ppo_update(policy, optimizer, r, advantages, returns)

        history.append({
            "step": step,
            "rm_score": ((r["scores"] - RM_MEAN) / RM_STD).mean().item(),
            "kl": kl.mean().item(),
            "length": r["mask"].sum(dim=1).mean().item(),
            **stats,
        })
        if step % print_every == 0 or step == num_steps - 1:
            h = history[-1]
            print(f"step {step:4d}  reward {h['rm_score']:+.3f}  KL {h['kl']:6.2f}  "
                  f"len {h['length']:5.1f}  clip {h['clip_frac']:.2f}")
            print(f"      {r['prompts'][0][:60]!r} -> {r['responses'][0][:100]!r}")
    return policy, history

### Train with the KL penalty

`KL_COEF = 0.05` is a common starting point. Watch two numbers:

- **reward** (normalised RM score, 0 = average SFT response) should climb.
- **KL** (summed over the response, in nats) should grow slowly and level off. The penalty is doing its job when it stays in the low single digits.

In [ ]:
KL_COEF = 0.05
PPO_STEPS = 150

policy_kl, history_kl = run_ppo(kl_coef=KL_COEF, num_steps=PPO_STEPS)

### Experiment: remove the KL penalty

Same run, `KL_COEF = 0`, and a bit longer. Now nothing ties the policy to the SFT model, so it's free to chase whatever the reward model happens to like. Typical symptoms: the reward keeps climbing, KL grows without bound, and samples collapse into repetitive or odd text that scores well but reads badly. How fast this happens varies by seed and step count. If it still looks fine, raise `NO_KL_STEPS`.

In [ ]:
NO_KL_STEPS = 250

policy_nokl, history_nokl = run_ppo(kl_coef=0.0, num_steps=NO_KL_STEPS)

In [ ]:
# Reward and KL on separate charts (different units, so never a shared axis).
# Colours: categorical slot 1 (blue) = with KL, slot 2 (orange) = without; 2px lines; legend + recessive grid.
BLUE, ORANGE, INK = "#2a78d6", "#eb6834", "#52514e"
runs = [(f"with KL (β={KL_COEF})", history_kl, BLUE), ("no KL (β=0)", history_nokl, ORANGE)]

fig, axes = plt.subplots(1, 3, figsize=(15, 3.8))
for ax, key, title in zip(axes, ["rm_score", "kl", "length"],
                          ["Reward model score (normalised)", "KL from SFT (nats per response)", "Response length (tokens)"]):
    for label, hist, color in runs:
        ax.plot([h["step"] for h in hist], smooth([h[key] for h in hist], window=10), color=color, linewidth=2, label=label)
    ax.set_title(title)
    ax.set_xlabel("PPO step")
    ax.grid(alpha=0.25)
    ax.spines[["top", "right"]].set_visible(False)
    ax.tick_params(colors=INK)
axes[0].axhline(0, color=INK, linewidth=1, linestyle="--")
axes[0].legend(frameon=False)
plt.tight_layout()
plt.show()

## Compare every stage

Greedy responses to the same prompts at every stage, with the reward model's (normalised) score. Remember the score is the reward model's opinion, not the truth. The no-KL column is where the two can disagree most.

In [ ]:
policy_kl.eval()
policy_nokl.eval()
for p in test_prompts:
    results[p]["ppo"] = generate(policy_kl.lm, p)
    results[p]["ppo_no_kl"] = generate(policy_nokl.lm, p)

stages = [("base", "BASE"), ("sft", "SFT"), ("ppo", f"PPO + KL (β={KL_COEF})"), ("ppo_no_kl", "PPO, no KL")]
for p in test_prompts:
    scores = rm_score([p] * len(stages), [results[p][k] for k, _ in stages])
    print("=" * 100, "\nINSTRUCTION:", p)
    for (key, name), s in zip(stages, scores.tolist()):
        print(f"\n--- {name}  [reward {(s - RM_MEAN) / RM_STD:+.2f}]")
        print(results[p][key])

Try your own instruction:

In [ ]:
instruction = "Explain what a neural network is to a ten year old."
for name, model in [("SFT", ref_model), ("PPO + KL", policy_kl.lm), ("PPO, no KL", policy_nokl.lm)]:
    print(f"--- {name}\n{generate(model, instruction)}\n")

## Takeaways

- **SFT** teaches format by imitation, but can only be as good as the demonstrations.
- **The reward model** learns human preferences from comparisons. It's cheap to label and generalises to any (prompt, response), but it's an imperfect proxy (≈60 to 70% accurate here).
- **PPO** optimises the policy against that proxy. The **KL penalty against the frozen SFT model** is what makes this safe: it bounds how far the policy can drift, which limits reward hacking and keeps the policy where the reward model can be trusted.
- In the no-KL run, the reward model's score and actual quality come apart: that gap is reward hacking.

Newer methods such as **DPO** skip the explicit reward model and RL loop: they optimise the same KL-regularised objective directly on preference pairs. The KL-to-reference idea is still at the centre.

In [ ]:
policy_kl.lm.save_pretrained("ppo-gpt2")
tokenizer.save_pretrained("ppo-gpt2")